In [1]:
from collections import defaultdict
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
)
from context_flux_no.models.multiphysics.hyperfluxfno import (
    HyperNeuralOperator,
)
from context_flux_no.training.io import load_model
from context_flux_no.utils import num_parameters
from einops import rearrange
from jaxtyping import Array, Float, PRNGKeyArray
from tqdm import tqdm


jax.config.update("jax_default_device", jax.devices("gpu")[3])

In [ ]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FluxNO",
    target_network_kwargs=dict(
        stencil_widths=(10, 10), lift_dim=128, hidden_dim=128, depth=4
    ),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)
# TODO: make this more ergonomic
dummy_target = hyperfluxno.hypernetwork_head(
    hyperfluxno.hypernetwork_trunk(
        jnp.zeros(
            hyperfluxno.embedding_dim,
        )
    )
)
print(num_parameters(dummy_target) / 1e6)

/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/hyperfluxfno/utils.py:53: UserWarning: TRecViTEncoder supports variable in_timesteps. The given 
                    in_timesteps value will be ignored.
  warnings.warn(


In [ ]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FNO",
    target_network_kwargs=dict(
        frequency_modes=8, lift_dim=48, depth=4, width_lift=48, width_project=48
    ),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)
dummy_target = hyperfluxno.hypernetwork_head(
    hyperfluxno.hypernetwork_trunk(
        jnp.zeros(
            hyperfluxno.embedding_dim,
        )
    )
)
print(num_parameters(dummy_target) / 1e6)

3.217636
0.088033


In [ ]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="UNet",
    target_network_kwargs=dict(hidden_channels_base=8, groups_norm=4, stack_grid=True),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)
dummy_target = hyperfluxno.hypernetwork_head(
    hyperfluxno.hypernetwork_trunk(
        jnp.zeros(
            hyperfluxno.embedding_dim,
        )
    )
)
print(num_parameters(dummy_target) / 1e6)

4.01122
0.112081


In [11]:
@eqx.filter_jit
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


test_data = jax.random.normal(jax.random.key(0), (512, 21, 1, 100))

In [12]:
hyperfluxno(test_data[0, :20], (0.1, 0.01))

(20, 2, 100)


E0710 01:24:07.247439 3344893 xtile_compiler.cc:399] Fusion: gemm_fusion_dot = f32[128,500]{1,0} fusion(a.1, bitcast.14), kind=kCustom, calls=gemm_fusion_dot_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0710 01:24:07.247536 3344893 xtile_compiler.cc:401] Computation: gemm_fusion_dot_computation.clone {
  parameter_0 = f32[128,128]{1,0} parameter(0)
  parameter_1 = f32[128,500]{0,1} parameter(1)
  ROOT dot.1 = f32[128,500]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0710 01:24:07.252341 3344857 xtile_compiler.cc:399] Fusion: gemm_fusion_do

(Array([[ 0.3117354 ,  1.5995171 ,  1.161767  ,  0.30145848,  0.71029204,
          2.2914884 , -0.5496999 , -0.5576248 ,  1.205263  ,  0.31447637,
          0.10655051,  0.60784316,  0.9410156 , -1.0602068 ,  0.95167434,
          1.2393031 ,  0.4524158 ,  0.5825163 , -0.01213515, -0.556567  ,
          0.3383975 ,  1.8390743 , -0.07142672, -0.83162093, -0.08063704,
         -0.90451646,  0.21428166, -0.48618174, -0.7169968 ,  0.00637545,
          1.0367059 , -0.71844435, -0.30706817, -1.0998288 ,  0.6573541 ,
         -0.81378806,  0.0200156 ,  2.0252435 ,  0.0799599 ,  1.2030272 ,
          1.1551961 ,  0.06607585,  0.07930022, -0.77886766, -0.99046254,
         -0.05651702,  0.25248617, -1.255871  , -0.2361033 , -0.7890959 ,
         -0.01263246,  0.94700885,  1.908508  , -0.8725474 ,  1.7357244 ,
         -0.62778145, -1.7014449 , -0.9022924 , -0.94335294, -1.3628522 ,
         -0.97977024,  0.7835078 ,  1.455822  , -1.0197797 , -1.0203825 ,
          0.6053051 ,  0.11628982, -0.

In [13]:
loss_fn(hyperfluxno, test_data, (0.1, 0.01), jax.random.key(0))

(20, 2, 100)


E0710 01:24:29.279987 3344839 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.28 = f32[512,128]{1,0} fusion(mul.323, dynamic_nodonate__first___73_.1), kind=kCustom, calls=gemm_fusion_dot.28_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0710 01:24:29.280087 3344839 xtile_compiler.cc:401] Computation: gemm_fusion_dot.28_computation.clone {
  parameter_0.27 = f32[512,128]{1,0} parameter(0)
  parameter_1.27 = f32[128,128]{1,0} parameter(1)
  ROOT dot.67 = f32[512,128]{1,0} dot(parameter_0.27, parameter_1.27), lhs_contracting_dims={1}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0710 01:24:29.287892 334486

(Array(2.175629, dtype=float32), {})

## Evaluate the trained target networks

In [5]:
model_paths = {
    "FNO": [
        "seed=0/26-07-10-01:51:12",
        "seed=10/26-07-10-01:34:31",
        "seed=20/26-07-10-01:34:31",
    ],
    "UNet": [
        "seed=0/26-07-10-01:39:05",
        "seed=10/26-07-10-01:39:05",
        "seed=20/26-07-10-01:39:05",
    ],
}

In [2]:
datadir = Path("../../data")
dataset_test = (
    xr.open_dataset(
        datadir
        / "datasets/cubic_no_source/data/test/cubic_no_source_large_test_seed=10.hdf5",
        engine="h5netcdf",
        chunks={},
    )
    .isel(t=slice(None, None, 10))
    .isel({"t": slice(0, 99)})
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

(600000, 40, 1, 100)

In [3]:
@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }


In [6]:
results_dict = defaultdict(list)
for target_type, paths in model_paths.items():
    for p in tqdm(paths):
        loaddir = Path(
            "../../checkpoints/cubicflux_1d/HyperNeuralOperator/OneStepLoss/"
        )
        model = load_model(loaddir / p)
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=6000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[target_type].append(jax.tree.map(jnp.mean, results))


  0%|          | 0/3 [00:00<?, ?it/s]/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/hyperfluxfno/utils.py:53: UserWarning: TRecViTEncoder supports variable in_timesteps. The given 
                    in_timesteps value will be ignored.
  warnings.warn(
/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/nn/structured_linear.py:40: UserWarning: out_features is not divisible by num_blocks. Output vector 
            will be truncated to the requested size.
  warnings.warn("""out_features is not divisible by num_blocks. Output vector


(20, 2, 100)


E0716 15:50:27.045791 2712332 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0716 15:50:27.218662 2712332 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0716 15:50:27.394503 2712332 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0716 15:50:27.573798 2712332 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0716 15:50:28.531214 2712332 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0716 15:50:50.989315 2712332 cuda_timer.cc:8

{'l2_onestep': (600000,), 'l2_rollout': (600000,), 'l_inf_onestep': (600000,), 'l_inf_rollout': (600000,)}


 67%|██████▋   | 2/3 [53:17<26:35, 1595.54s/it]

...

{'l2_onestep': (600000,), 'l2_rollout': (600000,), 'l_inf_onestep': (600000,), 'l_inf_rollout': (600000,)}


 67%|██████▋   | 2/3 [55:11<27:25, 1645.35s/it]

{'l2_onestep': (600000,), 'l2_rollout': (600000,), 'l_inf_onestep': (600000,), 'l_inf_rollout': (600000,)}


100%|██████████| 3/3 [1:21:27<00:00, 1629.12s/it]

{'l2_onestep': (600000,), 'l2_rollout': (600000,), 'l_inf_onestep': (600000,), 'l_inf_rollout': (600000,)}


In [7]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    print(
        model_name,
        jax.tree.map(
            lambda list_: {
                "mean": jnp.mean(jnp.asarray(list_)),
                "std": jnp.std(jnp.asarray(list_)),
            },
            res_,
            is_leaf=lambda x: isinstance(x, list),
        ),
    )

FNO {'l2_onestep': {'mean': Array(0.01614738, dtype=float32), 'std': Array(0.0005241, dtype=float32)}, 'l2_rollout': {'mean': Array(0.15287593, dtype=float32), 'std': Array(0.00502522, dtype=float32)}, 'l_inf_onestep': {'mean': Array(0.05653831, dtype=float32), 'std': Array(0.0023002, dtype=float32)}, 'l_inf_rollout': {'mean': Array(0.7207095, dtype=float32), 'std': Array(0.01219429, dtype=float32)}}
UNet {'l2_onestep': {'mean': Array(0.0074907, dtype=float32), 'std': Array(0.00055443, dtype=float32)}, 'l2_rollout': {'mean': Array(0.08296157, dtype=float32), 'std': Array(0.00701032, dtype=float32)}, 'l_inf_onestep': {'mean': Array(0.02739722, dtype=float32), 'std': Array(0.00236788, dtype=float32)}, 'l_inf_rollout': {'mean': Array(0.5133549, dtype=float32), 'std': Array(0.03486236, dtype=float32)}}
